In [1]:
from math import *
import numpy as np

import pandas as pd
import requests
import io

from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time
from astropy.time import TimeDelta

from scipy.optimize import curve_fit

from astropy import units as u
from astropy.coordinates import SkyCoord

import glob
import os
import re

import matplotlib.pyplot as plt
import matplotlib
from matplotlib import cm
%matplotlib widget

from scipy.optimize import curve_fit

In [2]:
filename = './Neptune_R.fits'
hdul = fits.open(filename)
hdu = hdul[0]
hdr = hdu.header
img = (hdu.data).astype(float)

In [3]:
hdu.header

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 2072                                                  
NAXIS2  =                 1410                                                  
TARGET  = 'Neptune '                                                            
DATE-OBS= '2025-10-15T20:09:07.950358'                                          
DATE-END= '2025-10-15T20:09:13.436362'                                          
RA      = '00:01:25.9'                                                          
DEC     = '-01:20:29.8'                                                         
EXPTIME =                  5.0                                                  
FOCUS   =                 8150                                                  
FILTER  = 'R       '        

In [4]:
transform_mtx = np.array([[ hdr['CD1_1'], hdr['CD1_2'] ], \
                          [ hdr['CD2_1'], hdr['CD2_2'] ]])

x0, y0 = hdr['CRPIX1'], hdr['CRPIX2']
 
def angular_distance(ra,dec,RA,DEC):
    return acos(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))

def equatorial_to_tangential(ra,dec,RA,DEC):
    ksi = cos(dec)*sin(ra-RA)/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    eta = (sin(dec)*cos(DEC)-cos(dec)*sin(DEC)*cos(ra-RA))/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    return ksi,eta

def tangential_to_equatorial(ksi,eta,RA,Dec):
    x,y,z = np.dot(np.array([[-sin(RA),-cos(RA) * sin(Dec),cos(RA) * cos(Dec)],
                             [cos(RA),-sin(RA) * sin(Dec),sin(RA) * cos(Dec)],
                             [0,cos(Dec),sin(Dec)]]),
                   np.array([ksi,eta,1]))/sqrt(1+ksi*ksi+eta*eta)
    ra = atan2(y,x)
    dec = atan2(z,sqrt(x*x+y*y))
    if(ra<0):
        ra+=2*pi
    return ra,dec    

def tangential_to_pixel(xi, eta, transform=transform_mtx, x0=x0, y0=y0):
    x, y = np.dot(np.linalg.inv(transform), [xi, eta]) + np.array([x0, y0])
    return x, y

def pixel_to_tangential(x, y, transform=transform_mtx, x0=x0, y0=y0):
    xi, eta = np.dot(transform, [x-x0, y-y0])
    return xi, eta

def eq2pix(ra, dec, transform=transform_mtx, x0=x0, y0=y0):
    return tangential_to_pixel(*np.rad2deg( 
                                equatorial_to_tangential(*np.deg2rad((ra, dec, ra0, dec0)))), transform, x0, y0)

In [32]:
ra0, dec0 = float(hdr["CRVAL1"]), float(hdr["CRVAL2"])
ra0, dec0

(0.357928325886, -1.33952892983)

In [6]:
w, h = float(hdr["IMAGEW"]), float(hdr["IMAGEH"])

In [7]:
t = Time(hdr['DATE-OBS'], format='isot')
t+=TimeDelta(float(hdr['EXPTIME']), format='sec')/2
JD = float(t.copy(format='jd').value)
yr = float(t.copy(format='jyear').value) 


fov = 0.6

JD, fov

(2460964.339704287, 0.6)

In [8]:
def get_Gaia_data(yr, lm1,lm2, gdf):
    ra,dec,gmag = [],[],[]
    for index, row in gdf.iterrows():
        G = row['phot_g_mean_mag']
        if (not isnan(G)) and (lm1<G<lm2):
            ra0,dec0 = \
                radians(row['ra']), \
                radians(row['dec'])
            ra.append(ra0)
            dec.append(dec0)
            gmag.append(G)
    return np.array(ra),np.array(dec),np.array(gmag)

In [9]:
response = requests.get(f'http://sfa.puldb.ru:9810?cmd=box&ra={ra0}&dec={dec0}&fov={fov}')
df = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
df

,solution_id,designation,source_id,random_index,ref_epoch,ra,ra_error,dec,dec_error,parallax,...,azero_gspphot,azero_gspphot_lower,azero_gspphot_upper,ag_gspphot,ag_gspphot_lower,ag_gspphot_upper,ebpminrp_gspphot,ebpminrp_gspphot_lower,ebpminrp_gspphot_upper,libname_gspphot
0,1636148068921376768,Gaia DR3 2449459878004175872,2449459878004175872,1086431203,2016.0,0.472452,0.226896,-1.638709,0.186496,0.209480,...,0.0041,0.0009,0.0103,0.0033,0.0007,0.0083,0.0018,0.0004,0.0045,MARCS
1,1636148068921376768,Gaia DR3 2449459951019930880,2449459951019930880,1420671998,2016.0,0.452730,0.049380,-1.631613,0.036459,2.871135,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1636148068921376768,Gaia DR3 2449460054098203904,2449460054098203904,1590698675,2016.0,0.467420,1.141239,-1.618027,0.905428,-1.535211,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1636148068921376768,Gaia DR3 2449460088458893440,2449460088458893440,964508933,2016.0,0.488780,0.087346,-1.612004,0.060884,0.212263,...,0.0029,0.0007,0.0076,0.0025,0.0006,0.0065,0.0013,0.0003,0.0035,MARCS
4,1636148068921376768,Gaia DR3 2449460530840340480,2449460530840340480,1028960532,2016.0,0.400984,0.846564,-1.627496,0.724704,1.022427,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,1636148068921376768,Gaia DR3 2449820483459438080,2449820483459438080,135264752,2016.0,0.112150,1.946963,-1.049904,1.518777,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1151,1636148068921376768,Gaia DR3 2449820487754563072,2449820487754563072,824835402,2016.0,0.105248,0.352928,-1.059248,0.377186,2.202170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1152,1636148068921376768,Gaia DR3 2449820517819175808,2449820517819175808,600839264,2016.0,0.119751,3.356353,-1.049287,2.629505,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1153,1636148068921376768,Gaia DR3 2449820556474039296,2449820556474039296,1110403513,2016.0,0.092362,0.097714,-1.044730,0.082480,0.254036,...,0.0039,0.0008,0.0097,0.0033,0.0006,0.0081,0.0018,0.0003,0.0045,MARCS


In [16]:
limmag1,limmag2 = 6.0,16.0
#limmag1,limmag2 = -1000.0,1000.0

ra, dec, gmag = get_Gaia_data(yr,limmag1,limmag2, df)
len(ra), len(dec), len(gmag)

(130, 130, 130)

In [17]:
W = WCS(hdul[0].header)

bdr = 50
pxpos = W.wcs_world2pix(np.stack(np.rad2deg((ra, dec)), axis=-1),1)  
print(pxpos)

p_indx = np.where((pxpos[:,0]>bdr) & (pxpos[:,1]>bdr) & (pxpos[:,0]<w-bdr) & (pxpos[:,1]<h-bdr))

pxpos = pxpos[p_indx]
ra,dec,gmag = ra[p_indx],dec[p_indx],gmag[p_indx]

x = [p[0] for p in pxpos]
y = [p[1] for p in pxpos]

print(len(pxpos))

[[  208.63610221  2724.56539224]
 [  126.04662307  2635.26123953]
 [ 1350.44418555  2720.53211755]
 [ 1668.13804384  2902.76549838]
 [ 1099.3686071   2783.7841637 ]
 [ 1087.28654833  2783.84027522]
 [  713.93400702  2289.11408791]
 [ 1175.3946514   2189.59101723]
 [ 1065.71042694  1972.40123008]
 [ 3016.96626973  2933.09709498]
 [ 2838.77979922  2746.1288113 ]
 [ 2114.25420129  2717.02891724]
 [ 2030.49607154  2488.6796956 ]
 [ 1981.20267257  2371.33627969]
 [ 2093.84407657  2215.23158789]
 [ 2531.00833705  2286.24264293]
 [ 2643.25166656  2303.3423466 ]
 [ 2674.18084841  2141.71637035]
 [ 2060.31885981  2144.7730475 ]
 [ 1953.88580749  1695.47256923]
 [ 3033.98640983  2213.76084164]
 [ 3017.36232013  2024.67511785]
 [ 2726.86098083  2093.10253321]
 [ 2194.93161741  1470.76164882]
 [ 2161.76434769  1347.00566002]
 [ 2053.18763756  1061.43560364]
 [ 2666.94066065  1170.73885055]
 [ 2751.29730008  1159.85010981]
 [ 3046.20830043   978.85846891]
 [ 2596.54565077   670.47951533]
 [ 2829.99

In [18]:
from astropy.stats import sigma_clip

img_processed = sigma_clip(img, sigma_lower=5.0, sigma_upper=10.0)
fig, ax = plt.subplots(figsize=(3.0,3.0), dpi=300)
ax.xaxis.set_tick_params(labelsize=5)
ax.yaxis.set_tick_params(labelsize=5)
plt.tight_layout()

ax.imshow(img_processed, origin='lower', cmap="gray") 
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [19]:
size = 15
line_width = 0.2
ax.scatter(x, y, marker='s', facecolors='none', edgecolors='red', s=size, lw=line_width)

In [20]:
def gaussian2D(box,*p):
    x = np.arange(0, box, 1)
    y = np.arange(0, box, 1)
    x, y = np.meshgrid(x,y)
    Ibkg,Imax,A,B,C,xc,yc = p
    r = A*(x-xc)**2+C*(y-yc)**2+B*(x-xc)*(y-yc)
    return np.ravel(Ibkg+Imax*np.exp(-r))

def moffat(xy, *p):
    x, y = xy
    Ibkg, Imax, alpha, beta, xc, yc = p
    r2 = (x - xc)**2 + (y - yc)**2
    return Ibkg + Imax * (1 + r2 / alpha**2) ** (-beta)

def im_centroiding(I):
    h, w = I.shape

    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    xdata = np.shape(I)[0]
    ydata = I.ravel()

    # gaussian
    Ib = np.median(I)
    Im = np.max(I)
    A = C = 1/5
    B = 0.0
    xc0 = w // 2
    yc0 = h // 2
    assert(xc0 == yc0)
    popt, pcov = curve_fit(f=gaussian2D, p0=[Ib, Im, A, B, C, xc0, yc0],
                           xdata=xdata, ydata=ydata, maxfev=1000)
    
    #alpha = (h+w)/8.0
    #beta = 2.0
    #popt, pcov = curve_fit(f=moffat, p0=[Ib, Im, alpha, beta, xc0, yc0],
    #                       xdata=xdata, ydata=ydata, maxfev=10000)
    return popt, pcov

In [33]:
ap = size
xp,yp = [],[]
xi,eta = [],[]
num_success = num_failure = num_accepted = 0

i=0
for px, py in (pxpos).astype(int):
    I = img[py-ap:py+ap,px-ap:px+ap]
    #print(I.shape)
    #print(I.ravel().shape)
    try:
        p, covp = im_centroiding(I)
        num_success += 1
        print((px,py), (p[-2]-ap+px, p[-1]-ap+py), "\tRegression success")
    except:
        num_failure += 1
        print((px,py), "\tRegression failure")
        continue
    
    xc = p[-2]
    yc = p[-1]
    threshold = 0.25*0.25
    print(sqrt(covp[-1,-1]), sqrt(covp[-2,-2]))
    if 0.0<=covp[-1,-1]<threshold and 0.0<=covp[-2,-2]<threshold:
        num_accepted += 1
        xp.append(px-ap+xc)
        yp.append(py-ap+yc)
        
        tx,ty = equatorial_to_tangential(ra[i], dec[i], radians(ra0), radians(dec0))
        xi.append(tx)
        eta.append(ty)
    i+=1
print(f"number of failures: {num_failure}\nnumber of successes: {num_success}\nof those accepted: {num_accepted}")

(257, 1141) (257.6035020799547, 1141.461901272101) 	Regression success
0.03505524615766812 0.0249165340930497
(1334, 1349) (1334.6645627862783, 1346.5323143662745) 	Regression success
0.42327578020170026 0.21007899209516476
(1650, 1344) (1650.411505760413, 1343.3455669342063) 	Regression success
0.025098839965867734 0.03192204815126756
(1212, 854) (1212.319040248994, 854.5408778261816) 	Regression success
0.05254308985296761 0.04240913740088385
(148, 1014) (148.70236518553418, 1014.6280554209054) 	Regression success
0.14170954410849945 0.09987211722927816
(319, 867) (319.4934173513715, 867.4589167981744) 	Regression success
0.10235316809338406 0.07954766796489202
(1386, 822) (1386.257225087285, 822.3697796189447) 	Regression success
0.10889066078812593 0.0979005288747979
(1541, 835) (1540.5573862700937, 834.3960166500623) 	Regression success
0.013259311734589324 0.012162428525799246
(1808, 345) (1807.4329205970669, 345.9114866152386) 	Regression success
0.028139033577383645 0.027893838

In [34]:
img_processed = sigma_clip(img, sigma_lower=5.0, sigma_upper=10.0)
fig, ax = plt.subplots(figsize=(3.0,3.0), dpi=300)
ax.xaxis.set_tick_params(labelsize=5)
ax.yaxis.set_tick_params(labelsize=5)
plt.tight_layout()

ax.imshow(img_processed, origin='lower', cmap="gray") 
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [35]:
size = 15
line_width = 1.0
ax.scatter(xp, yp, marker='o', facecolors='none', edgecolors='red', s=size, lw=line_width)

In [36]:
Lxi = np.rad2deg(xi)*3600
Leta = np.rad2deg(eta)*3600
C = np.column_stack((xp, yp, np.ones_like(xp)))
N = len(xp)

In [37]:
C.astype(int)

array([[ 257, 1141,    1],
       [1650, 1343,    1],
       [1212,  854,    1],
       [ 148, 1014,    1],
       [ 319,  867,    1],
       [1386,  822,    1],
       [1540,  834,    1],
       [1807,  345,    1],
       [1189,  170,    1],
       [  96,  234,    1]])

In [38]:
Zx = np.dot(np.linalg.inv(np.dot(C.T, C)), np.dot(C.T, Lxi))
Zy = np.dot(np.linalg.inv(np.dot(C.T, C)), np.dot(C.T, Leta))

r_xi = Lxi - np.dot(C,Zx)
r_eta = Leta - np.dot(C,Zy)

uwe_xi = sqrt(np.dot(r_xi.T,r_xi)/(N-3)) * u.arcsec
uwe_eta = sqrt(np.dot(r_eta.T,r_eta)/(N-3)) * u.arcsec
uwe_xi.to(u.arcsec), uwe_eta.to(u.arcsec)

(<Quantity 0.09413576 arcsec>, <Quantity 0.12512943 arcsec>)

In [39]:
r_xi

array([-0.10033196,  0.04632236,  0.00632115,  0.15375097, -0.03702564,
       -0.02307606, -0.05658129, -0.04917146,  0.12162654, -0.0618346 ])

In [40]:
r_eta

array([ 0.06138886, -0.1303397 ,  0.09955558,  0.12681586, -0.0641954 ,
        0.06779156, -0.11716992,  0.04392677,  0.08768794, -0.17546154])

In [41]:
Lxi

array([ 376.06427604, -305.15142041,  -87.26704146,  430.53526148,
        348.25245447, -171.90010422, -247.32958026, -373.40402018,
        -70.27791634,  462.44096363])

In [42]:
Leta

array([-219.36003959, -307.14036023,  -71.46471666, -158.16369438,
        -85.05547204,  -54.38509178,  -59.21948287,  181.84680214,
        262.804056  ,  222.43674649])